# Demo: Ingest a NOMAD entry

This notebook demonstrates fetching a NOMAD entry (live or mocked) and ingesting
it with the pipeline to preserve raw payloads and produce a normalised MaterialRecord.
The `NOMADAPIAdapter` supports an injected client for testing or will perform an HTTP
GET against the NOMAD public API when run in an environment with network access.

In [ ]:
# Imports and setup
from pathlib import Path

from materials_pipeline.pipeline import IngestionPipeline
from materials_pipeline.providers.nomad import NomadProvider
from materials_pipeline.providers.nomad_api import NOMADAPIAdapter

# Set the project root to the repository root (adjust if running elsewhere)
notebook_root = Path(".").cwd()
project_root = notebook_root.parent
print("Project root:", project_root)

In [ ]:
# Create the adapter and pipeline. Pass `client=` to NOMADAPIAdapter for a fake client
adapter = NOMADAPIAdapter()
pipeline = IngestionPipeline(project_root=project_root)
provider = NomadProvider()  # use NOMAD-specific provider for demo

# Example NOMAD entry id (public example)
entry_id = "w5hdgY4N38hhf6LMdQxy0niDxBMh"
print("Will fetch entry:", entry_id)

In [ ]:
# Fetch the provider-native record (network request unless a client was injected).
raw = adapter.fetch_entry(entry_id)
print("Raw keys:", list(raw.keys()))
print("formula_pretty =", raw.get("formula_pretty"))
print("chemsys =", raw.get("chemsys"))

In [ ]:
# Ingest the record using the pipeline
result = pipeline.ingest(provider=provider, raw=raw)
print("Ingestion status:", result.status)
if result.raw_path:
    print("Raw saved to:", result.raw_path)
if result.normalised_path:
    print("Normalised saved to:", result.normalised_path)

if result.material is not None:
    md = result.material.model_dump()
    print("Reduced formula:", md["composition"]["reduced_formula"])
    print("Element amounts:", md["composition"]["element_amounts"])

## Notes
- For repeatable demos or CI inject a fake client as in tests/test_nomad_api_adapter.py.
- The adapter uses `requests` for HTTP calls when no client is provided.